In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 0. Environment Setup

# Clone the gemma repository
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository for UI/UX
!git clone https://github.com/google-deepmind/dialog.git || true
!pip install -q ./dialog

# Ensure local modules are in path
import sys
import os
sys.path.append(os.getcwd())

In [ ]:
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

In [ ]:
import importlib
import jax
import os
import gemma_titans
# importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils
from gemma import gm

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"

# jax.config.update("jax_disable_jit", True) # Temporarily disable JIT to bypass the hashing error

In [ ]:
import dataclasses

titans_phase2_first_layer = 11  # Titans layers from this index onward are active.
                                 # Earlier layers revert to standard Gemma blocks.
                                 # 17 → layers 17,23 active (~25GB compile RAM)
                                 # 23 → layer 23 only  (~5GB compile RAM)

_all_titans_layers = (5, 11, 17, 23)
active_titans_layers = tuple(l for l in _all_titans_layers if l >= titans_phase2_first_layer)
print(f"Active Titans layers: {active_titans_layers}")

_override_keys = {'is_training_mode', 'titans_layer_indices', 'titans_phase2_first_layer'}
inference_config = Gemma_Titans_Config(
    **{f.name: getattr(Gemma3_1B_Titans.config, f.name) for f in dataclasses.fields(Gemma_Titans_Config) if f.name not in _override_keys},
    is_training_mode=False,
    titans_layer_indices=active_titans_layers,
    titans_phase2_first_layer=titans_phase2_first_layer,
)

In [ ]:
import google.colab
import shutil
import orbax.checkpoint as ocp

zip_path = '/content/drive/Shareddrives/shared veriga/saved_titans_delta/saved_titans_phase2_from_11.zip'
if os.path.exists(zip_path):
    shutil.unpack_archive(zip_path, "./saved_titans_delta")
    print(f"Unpacked {zip_path}")
else:
    print(f"WARNING: zip not found at {zip_path}")

import jax.numpy as jnp

model = Gemma3_1B_Titans(
    config=inference_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.tokens",
)

print("Loading weights...")

def load_titans_weights(load_dir: str):
    checkpointer = ocp.StandardCheckpointer()
    return checkpointer.restore(os.path.abspath(load_dir))

titans_delta_path = "./saved_titans_delta"

print("Loading Gemma base weights...")
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

print("Loading Phase 2 Titans weights...")
loaded_titans_params = load_titans_weights(titans_delta_path)

active_layer_keys = {f'layer_{l}' for l in active_titans_layers}
loaded_titans_params = {k: v for k, v in loaded_titans_params.items() if k in active_layer_keys}
print(f"Merging Titans weights for: {sorted(active_layer_keys)}")

merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)

assert 'memory_gate' in merged_params['layer_11'], "FATAL: Titans weights failed to merge!"
assert 'memory_gate' not in merged_params['layer_5'], "FATAL: layer_5 must be standard Gemma!"

print("✅ Weights merged successfully!")

tokenizer = gm.text.Gemma3Tokenizer()

In [ ]:
merged_params['layer_5'].keys()

## 4. Interactive Dialogue with `google-deepmind/dialog`

In [ ]:
import dialog
from gemma import gm
import jax

# Initialize Sampler and Conversation
sampler = gm.text.Sampler(
    model=model,
    params=merged_params,
    tokenizer=tokenizer,
)

conv = dialog.Conversation()


def chat(user_input: str):
    global conv
    # Add user message
    conv += dialog.User(user_input)

    # Convert conversation to prompt (Gemma 3 format)
    prompt = conv.as_text(training=False)

    # Generate response
    # Note: Sampler handles the caching of Titans memory automatically
    response_text = sampler.sample(prompt, max_new_tokens=128)

    # Add model response to UI
    conv += dialog.Model(response_text)
    conv.show()

# Example usage:
# chat("Привет! Кто такие Титаны в мифологии?")

In [ ]:
chat("Who is Polyphemus in the ancient greek mythology?")
